# Seattle, WA — 50% + \$200k Building Exemption on Voted, Seattle-Contained Levies

**Center for Land Economics — LVTShift**

This model isolates the **voted property-tax levies whose taxing district is *wholly contained*
within the City of Seattle** and applies a **stacking building (improvement) exemption** to that
slice: the **first \$200,000** of every building's taxable value is fully exempt, and **50%** of
the remaining building value is exempt. A single millage is then set on the remaining
(land + un-exempted building) value so the targeted levies raise the same total — revenue-neutral
*within the targeted slice*, redistributed off buildings and onto land.

**Policy parameters (refined 2026-06-15; condo aggregation added 2026-07-23):**

| Decision | Choice |
|---|---|
| Reform | **Building exemption: \$200k base + 50%** of remainder (toolkit `model_stacking_improvement_exemption`, `floor=200_000`, `pct=0.50`) |
| Scope | **Voted levies wholly contained within the City of Seattle** — the city excess lid-lifts, the regular lid-lift, the voted G.O. bond, and the **Seattle Metropolitan Park District**. Combined rate **2.08119 /\$1000**. |
| Excluded | **Seattle School District No. 1** (≈ coextensive with the city but *not* wholly contained), plus all state / county / port / EMS / flood / Sound Transit levies. |
| Frame | **Isolate the targeted levies** — `current_tax`/`new_tax` are the targeted slice, not the full bill |
| Existing exemptions | **Preserved** — King County post-exemption taxable values; fully-exempt parcels held out of the solver |
| Condominiums | **Aggregated to the land parcel** (Section 1c) — each complex's unit-account taxable values are summed onto its `PROPTYPE 'K'` land parcel and the complex is modeled as one parcel, categorized by unit count (5+ → Large Multi-Family, 2–4 → Small Multi-Family) |
| Revenue target | **Derived from King County** 2025 *Codes and Levies* rate book × the parcel taxable base |

**Data:** King County GIS parcel layer + Assessor values, tax year **2025** (cached extract
`data/seattle_parcels_2025_11_20.parquet`). **Commercial building values are corrected** (Section 1b):
the raw roll carries a ~\$1,000 placeholder improvement value for many income-producing commercial
parcels (value held off-parcel on leasehold / personal-property accounts), so they are revalued by
the **cost approach** — Commercial Building file square footage × observed \$/sqft of comparable
valued buildings (taxable parcels only). This is **computed inline in Section 1b**, which downloads
the King County Assessor *Real Property Account* and *Commercial Building* files once and caches them
under `data/assessor/`. **Condominium unit values are aggregated to their complex's land parcel**
(Section 1c) from the same *Real Property Account* file. Levy rates: King County Assessor *2025
Codes and Levies* rate book (`data/levy/ratebook25.pdf`); levy classification per the millage
breakdown in `analysis/reports/seattle/millages/`.

In [ ]:
import sys
import json
import os
from pathlib import Path

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_stacking_improvement_exemption,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'seattle'
STATE_FIPS = '53'              # Washington
COUNTY_FIPS = '033'            # King County
MODEL_TYPE = 'building_abatement_50pct_200k'

IMPROVEMENT_EXEMPTION_PCT = 0.50    # 50% of building value above the base is exempt
BUILDING_ABATEMENT_FLOOR = 200_000  # first $200k of each building fully exempt

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

## Section 1 — Load Parcel Data

The cached extract is the King County GIS parcel layer joined to Assessor values
(tax year 2025). We filter to parcels whose **levy jurisdiction is the City of Seattle**
(`LEVY_JURIS == 'SEATTLE'`) — the set subject to City of Seattle levies — **plus the
condominium complex land parcels** (`PROPTYPE == 'K'`), which carry a null `LEVY_JURIS`
and are instead selected by Seattle tax code area. Each parcel's `LEVYCODE` (tax code
area) determines its exact levy composition.

In [ ]:
PARCEL_PATH = DATA_DIR / 'seattle_parcels_2025_11_20.parquet'

if PARCEL_PATH.exists():
    gdf = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(gdf):,} King County parcels from cache")
else:
    raise FileNotFoundError(
        "Expected cached King County parcel extract at "
        f"{PARCEL_PATH}. Source: King County GIS parcel layer + Assessor value "
        "extract (tax year 2025). Re-download from the KCGIS Open Data parcel "
        "service and join Assessor APPR/TAX value fields before re-running."
    )

# Filter to City of Seattle parcels (by levy jurisdiction), PLUS condominium complex land
# parcels: condo complexes carry PROPTYPE 'K' and a null LEVY_JURIS in the GIS layer, so the
# jurisdiction filter alone drops them. Pull them in by tax code area (the 8 Seattle TCAs);
# their values are rebuilt from the per-unit Assessor accounts in Section 1c.
SEATTLE_TCAS = {'0010', '0011', '0013', '0014', '0016', '0025', '0030', '0032'}
_is_city = gdf['LEVY_JURIS'] == 'SEATTLE'
_is_condo_complex = (gdf['PROPTYPE'] == 'K') & gdf['LEVYCODE'].isin(SEATTLE_TCAS)
gdf = gdf[_is_city | _is_condo_complex].copy()
print(f"City of Seattle parcels: {len(gdf):,} "
      f"(incl. {int(_is_condo_complex.sum()):,} condominium complex land parcels)")
print("Tax code areas present:", sorted(gdf['LEVYCODE'].dropna().unique().tolist()))

In [ ]:
# Section 1b — Restore reliable commercial building values (cost approach)
# The King County roll records a ~$1,000 PLACEHOLDER improvement value for many income-producing
# commercial parcels (retail, office, warehouse, restaurant, bank, ...) whose building value is
# carried off-parcel (leasehold / personal-property accounts). Left as-is they look like bare land
# and wrongly pin to the maximum increase. We restore those buildings by the COST APPROACH:
# building square footage (Assessor Commercial Building file) x the observed $/sqft of comparable
# Seattle buildings that DO have a real assessed value (taxable parcels only; exempt institutional
# excluded). Source files are downloaded once from the King County Assessor and cached locally.
import zipfile
import urllib.request

ASSESSOR_DIR = DATA_DIR / 'assessor'
ASSESSOR_DIR.mkdir(parents=True, exist_ok=True)
KC_ASSESSOR = 'https://aqua.kingcounty.gov/extranet/assessor/'
for _zip, _csv in [('Real Property Account.zip', 'EXTR_RPAcct_NoName.csv'),
                   ('Commercial Building.zip', 'EXTR_CommBldg.csv')]:
    if not (ASSESSOR_DIR / _csv).exists():
        print(f"Downloading {_zip} from King County Assessor ...")
        _dest = ASSESSOR_DIR / _zip
        urllib.request.urlretrieve(KC_ASSESSOR + _zip.replace(' ', '%20'), _dest)
        with zipfile.ZipFile(_dest) as _zf:
            _zf.extractall(ASSESSOR_DIR)

PLACEHOLDER_MAX = 1_000      # improvement values <= this are treated as placeholders
MIN_SQFT = 500               # a "real" building
PSF_LO, PSF_HI = 20, 2000    # trim implausible $/sqft when calibrating

# Real Property Account — appraised improvement + taxable status per parcel (PIN = Major+Minor)
_rp = pd.read_csv(ASSESSOR_DIR / 'EXTR_RPAcct_NoName.csv', encoding='latin-1',
                  usecols=['Major', 'Minor', 'TaxStat', 'ApprImpsVal'],
                  dtype={'Major': str, 'Minor': str})
_rp['PIN'] = _rp['Major'].str.zfill(6) + _rp['Minor'].str.zfill(4)
_rp['ApprImpsVal'] = pd.to_numeric(_rp['ApprImpsVal'], errors='coerce').fillna(0)
_rp['taxable'] = _rp['TaxStat'].astype(str).str.strip() == 'T'
_rp_pin = _rp.groupby('PIN').agg(rp_imp=('ApprImpsVal', 'sum'), any_taxable=('taxable', 'max'))

# Commercial Building — net (else gross) sqft + predominant use per parcel
_cb = pd.read_csv(ASSESSOR_DIR / 'EXTR_CommBldg.csv', encoding='latin-1',
                  usecols=['Major', 'Minor', 'PredominantUse', 'BldgNetSqFt', 'BldgGrossSqFt'],
                  dtype={'Major': str, 'Minor': str})
_cb['PIN'] = _cb['Major'].str.zfill(6) + _cb['Minor'].str.zfill(4)
for _c in ['BldgNetSqFt', 'BldgGrossSqFt', 'PredominantUse']:
    _cb[_c] = pd.to_numeric(_cb[_c], errors='coerce')
_cb['sqft'] = _cb['BldgNetSqFt'].where(_cb['BldgNetSqFt'] > 0, _cb['BldgGrossSqFt']).fillna(0)
_cbp = _cb.groupby('PIN').agg(sqft=('sqft', 'sum'),
                              use=('PredominantUse', lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan))

# Join onto Seattle parcels (gdf is already filtered to the city in Section 1)
gdf['PIN'] = gdf['PIN'].astype(str).str.zfill(10)
gdf['TAX_IMPR'] = pd.to_numeric(gdf['TAX_IMPR'], errors='coerce').fillna(0)
gdf['TAX_LNDVAL'] = pd.to_numeric(gdf['TAX_LNDVAL'], errors='coerce').fillna(0)
_j = (gdf[['PIN', 'TAX_IMPR', 'TAX_LNDVAL']]
      .merge(_cbp, left_on='PIN', right_index=True, how='left')
      .merge(_rp_pin, left_on='PIN', right_index=True, how='left'))
_j['sqft'] = _j['sqft'].fillna(0)

# Calibrate $/sqft on parcels that have BOTH a real assessed value and a real building
_cal = _j[(_j['rp_imp'] > 20_000) & (_j['sqft'] > MIN_SQFT)].copy()
_cal['psf'] = _cal['rp_imp'] / _cal['sqft']
_cal = _cal[(_cal['psf'] > PSF_LO) & (_cal['psf'] < PSF_HI)]
_psf_by_use = _cal.groupby('use')['psf'].median()
_psf_overall = float(_cal['psf'].median())

# Correction targets: placeholder improvement + a real building + TAXABLE (excludes exempt institutional)
_tgt = _j[(_j['TAX_IMPR'] <= PLACEHOLDER_MAX) & (_j['sqft'] > MIN_SQFT)
          & (_j['TAX_LNDVAL'] > 0) & (_j['any_taxable'] == True)].copy()
_tgt['imputed'] = (_tgt['sqft'] * _tgt['use'].map(_psf_by_use).fillna(_psf_overall)).round()
_imp_map = _tgt.set_index('PIN')['imputed']

gdf['imp_value_imputed'] = 0
_hit = gdf['PIN'].map(_imp_map)
_mask = _hit.notna() & (_hit > gdf['TAX_IMPR'])      # only raise placeholders, never lower a real value
_added = float((_hit[_mask] - gdf.loc[_mask, 'TAX_IMPR']).sum())
gdf.loc[_mask, 'TAX_IMPR'] = _hit[_mask].astype(float)
gdf.loc[_mask, 'imp_value_imputed'] = 1
print(f"Calibrated $/sqft on {len(_cal):,} comparable buildings (median ${_psf_overall:.0f}/sqft).")
print(f"Restored building values on {int(_mask.sum()):,} commercial parcels "
      f"(+${_added/1e9:.2f}B taxable improvement; cost-approach sqft x median $/sqft).")

### Section 1c — Condominium aggregation: unit accounts → land parcels

King County carries **every condo unit as its own Real Property account** (`Major` = the
condo plat/complex, `Minor` = the unit) with an allocated share of the complex's land value
plus the unit's improvement value. Units have **no polygon** in the GIS parcel layer; the
complex's land parcel *is* in the layer (`PROPTYPE 'K'`, `PIN = Major + '0000'`) but its
value fields hold only a **token amount ≈ one average unit** (median unit-sum ÷ K-record
ratio ≈ the complex's unit count), not the complex total — and its null `LEVY_JURIS`
excluded it from earlier runs entirely, leaving only n=44 misc. "Condominium" rows.

**Fix (Greg, 2026-07-23):** sum each complex's unit **taxable** land + improvement values
onto its land parcel and model the complex as a single parcel. Post-exemption taxable
values are used, so exempt units contribute \$0 and existing senior/nonprofit exemptions
are preserved. **Double-counting checks:** (1) the roll has *no* `Minor 0000` complex-level
account for these majors, so unit sums are the whole complex — the K record's token value
is *replaced*, never added; (2) any account that already appears as its own polygon in the
modeled frame is excluded from the unit sums.

**Exemption semantics:** the aggregated complex is one parcel row, so the \$200k building
floor applies **once per complex** (the model's per-parcel building-stack convention), not
once per unit.

In [ ]:
# Section 1c — Aggregate condominium unit accounts onto their complex land parcels.
# Unit accounts = Real Property accounts whose Major is a condo complex (a PROPTYPE 'K'
# polygon in the frame) and Minor != '0000'. Sums use post-exemption TAXABLE values.
_rpu = pd.read_csv(ASSESSOR_DIR / 'EXTR_RPAcct_NoName.csv', encoding='latin-1',
                   usecols=['Major', 'Minor', 'LevyCode', 'TaxStat',
                            'ApprLandVal', 'ApprImpsVal', 'TaxableLandVal', 'TaxableImpsVal'],
                   dtype={'Major': str, 'Minor': str, 'LevyCode': str})
_rpu['Major'] = _rpu['Major'].str.zfill(6)
_rpu['Minor'] = _rpu['Minor'].str.zfill(4)
_rpu['PIN'] = _rpu['Major'] + _rpu['Minor']
for _c in ['ApprLandVal', 'ApprImpsVal', 'TaxableLandVal', 'TaxableImpsVal']:
    _rpu[_c] = pd.to_numeric(_rpu[_c], errors='coerce').fillna(0)

_kmask = gdf['PROPTYPE'] == 'K'
_kmajors = set(gdf.loc[_kmask, 'PIN'].str[:6])

# Double-count guard 1: the complex itself must have NO account of its own in the roll
assert (_rpu['Major'].isin(_kmajors) & (_rpu['Minor'] == '0000')).sum() == 0, \
    "Minor-0000 complex accounts exist — unit sums would double-count"

_units = _rpu[_rpu['Major'].isin(_kmajors) & (_rpu['Minor'] != '0000')].copy()
# Double-count guard 2: drop accounts that already appear as their own polygon in the frame
_own_poly = _units['PIN'].isin(set(gdf['PIN']))
_units = _units[~_own_poly]
_units['exempt_unit'] = _units['TaxStat'].astype(str).str.strip() != 'T'

_cagg = _units.groupby('Major').agg(
    condo_units=('Minor', 'size'),
    exempt_units=('exempt_unit', 'sum'),
    t_land=('TaxableLandVal', 'sum'), t_imp=('TaxableImpsVal', 'sum'),
    a_land=('ApprLandVal', 'sum'), a_imp=('ApprImpsVal', 'sum'),
)

# Replace the K records' token values with the unit sums (never add — see guard 1)
_km = gdf.loc[_kmask, 'PIN'].str[:6]
_token_land = float(gdf.loc[_kmask, 'TAX_LNDVAL'].sum())
_token_imp = float(gdf.loc[_kmask, 'TAX_IMPR'].sum())
gdf['condo_units'] = np.nan
gdf.loc[_kmask, 'condo_units'] = _km.map(_cagg['condo_units'])
for _dst, _src in [('TAX_LNDVAL', 't_land'), ('TAX_IMPR', 't_imp'),
                   ('APPRLNDVAL', 'a_land'), ('APPR_IMPR', 'a_imp')]:
    gdf[_dst] = pd.to_numeric(gdf[_dst], errors='coerce').fillna(0)
    _v = _km.map(_cagg[_src])
    # a complex with no unit accounts (rare) keeps its token value rather than dropping to 0
    gdf.loc[_kmask, _dst] = _v.where(_v.notna(), gdf.loc[_kmask, _dst])

_no_units = int(_km.map(_cagg['condo_units']).isna().sum())
_agg_land = float(_units['TaxableLandVal'].sum())
_agg_imp = float(_units['TaxableImpsVal'].sum())
print(f"Condo complexes in frame: {int(_kmask.sum()):,} "
      f"({_no_units} with no unit accounts keep their token value; {int(_own_poly.sum())} "
      f"unit accounts skipped because they are already polygons in the frame)")
print(f"Unit accounts aggregated: {len(_units):,} "
      f"({int(_units['exempt_unit'].sum()):,} exempt units contributing $0 taxable)")
print(f"Taxable value added to the frame: ${_agg_land/1e9:.2f}B land + ${_agg_imp/1e9:.2f}B improvements")
print(f"  (replaces ${_token_land/1e6:.0f}M land + ${_token_imp/1e6:.0f}M improvements of token K-record value)")
_fully_exempt_cplx = int(((_cagg['t_land'] + _cagg['t_imp']) <= 0).sum())
print(f"Fully-exempt complexes (all units exempt): {_fully_exempt_cplx}")
print("Unit-count mix:", {'5+ units': int((_cagg['condo_units'] >= 5).sum()),
                          '2-4 units': int(_cagg['condo_units'].between(2, 4).sum()),
                          '1 unit': int((_cagg['condo_units'] == 1).sum())})

## Section 2 — Levy Classification & Targeted Current Tax

### Column mapping

| Concept | Column | Notes |
|---|---|---|
| Taxable land value | `TAX_LNDVAL` | Post-exemption (preserves existing exemptions) |
| Taxable improvement value | `TAX_IMPR` | Post-exemption building value |
| Appraised land / improvement | `APPRLNDVAL` / `APPR_IMPR` | Market value (reference) |
| Tax code area | `LEVYCODE` | Determines levy composition |
| Levy jurisdiction | `LEVY_JURIS` | City filter |
| Use description | `PREUSE_DESC` | Property categorization |

### Which levies are "voted *and* wholly contained within Seattle"?

From the King County Assessor **2025 Codes and Levies** rate book, cross-referenced with the
taxing-district boundary analysis in `analysis/reports/seattle/millages/`. The scope is the set
of voter-approved levies whose taxing **district boundary is exactly the City of Seattle** — so
the geography of the levy moves with the city, not a larger county/region/school district.

| Component | Rate /\$1000 | In scope? | Why |
|---|---|---|---|
| State School (Pt 1 + McCleary) | 2.24653 | ❌ | State, statewide |
| King County + its lid-lifts (Parks, Best Starts, VSHSL, Crisis Care) | 1.36182 | ❌ | Countywide |
| Port of Seattle | 0.10196 | ❌ | Countywide port district |
| Emergency Medical Services | 0.22146 | ❌ | Countywide district (EM-2 is only a sub-portion) |
| King County Flood District | 0.09757 | ❌ | Countywide |
| Sound Transit (RST1) | 0.16382 | ❌ | 3-county RTA |
| **Seattle School District No. 1** (EP&O + Capital/Bond) | 1.88116 | ❌ | Voted, but SD boundary only ≈ coextensive, **not wholly contained** |
| City of Seattle — regular (statutory) | 1.03867 | ❌ | Non-voted |
| **City of Seattle — excess lid-lifts** (Transportation, Housing, FEPP, Library, Democracy Voucher) | **1.57835** | ✅ | Voted, district = city |
| **City of Seattle — regular lid-lift** | **0.01970** | ✅ | Voter-approved lid-lift, district = city |
| **City of Seattle — voted G.O. bond** | **0.05459** | ✅ | Voted, district = city |
| **Seattle Metropolitan Park District (SMPD)** | **0.42855** | ✅ | Voted junior district, **coextensive with the city** |

**In-scope rate (uniform, all 8 Seattle TCAs):** 1.57835 + 0.01970 + 0.05459 + 0.42855 =
**2.08119 /\$1000**. Because the school levy is now out of scope, the rate no longer varies by
tax code area (the Highline-SD sliver in TCAs `0030`/`0032` is irrelevant here).

In [ ]:
# Official 2025 rate components (King County Assessor "Codes and Levies" rate book)
# IN SCOPE: VOTED property-tax levies whose taxing district is WHOLLY CONTAINED within the
# City of Seattle. Uniform across all 8 Seattle tax code areas (no school => no TCA variation).
CITY_LID_LIFT     = 1.57835   # voted excess lid-lifts (Transportation, Housing, FEPP, Library, Democracy Voucher)
CITY_REG_LID_LIFT = 0.01970   # voted regular lid-lift (folded into the regular levy)
CITY_GO_BOND      = 0.05459   # voted excess G.O. bond
SMPD_RATE         = 0.42855   # Seattle Metropolitan Park District (junior district = the city)
IN_SCOPE_RATE     = CITY_LID_LIFT + CITY_REG_LID_LIFT + CITY_GO_BOND + SMPD_RATE   # 2.08119 /$1000

# EXCLUDED: Seattle School District No. 1 (1.88116) — voted, but the SD boundary is only
# approximately coextensive with the city, not wholly contained, so it is out of scope.

# Total consolidated rate per TCA — for an all-levy reconstruction sanity check
TOTAL_RATE = {
    '0010': 9.19418, '0011': 9.19418, '0013': 9.19418,
    '0014': 9.19418, '0016': 9.19418, '0025': 9.19418,
    '0030': 11.27195, '0032': 11.27195,
}

# Clean taxable values (post-exemption)
gdf['taxable_land_value'] = pd.to_numeric(gdf['TAX_LNDVAL'], errors='coerce').fillna(0).clip(lower=0)
gdf['taxable_improvement_value'] = pd.to_numeric(gdf['TAX_IMPR'], errors='coerce').fillna(0).clip(lower=0)
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# In-scope (voted, wholly-contained) rate per parcel — uniform citywide
gdf['targeted_rate'] = IN_SCOPE_RATE
gdf['total_rate'] = gdf['LEVYCODE'].map(TOTAL_RATE)
assert gdf['total_rate'].notna().all(), "Unmapped LEVYCODE — check rate table"

# current_tax = the in-scope voted-levy slice only (isolate-the-levies frame)
gdf['current_tax'] = gdf['taxable_total'] * gdf['targeted_rate'] / 1000.0

# All-levy reconstruction (validation only)
recon_all = (gdf['taxable_total'] * gdf['total_rate'] / 1000.0).sum()
print(f"Total taxable value:                 ${gdf['taxable_total'].sum():,.0f}")
print(f"All-levy reconstructed Seattle tax:  ${recon_all:,.0f}  (rate x AV, official 2025 rates)")
print(f"IN-SCOPE slice (voted, Seattle-contained @ {IN_SCOPE_RATE:.5f}/$1000): ${gdf['current_tax'].sum():,.0f}")
print(f"  City lid-lifts + reg lid-lift + GO bond: ${(gdf['taxable_total']*(CITY_LID_LIFT+CITY_REG_LID_LIFT+CITY_GO_BOND)/1000).sum():,.0f}")
print(f"  Seattle Metropolitan Park District:      ${(gdf['taxable_total']*SMPD_RATE/1000).sum():,.0f}")

## Section 3 — Property Categories & Exempt Flag

Categories are built from the Assessor `PREUSE_DESC`. **Residential condominium complexes
(aggregated in Section 1c) are routed into the multifamily categories by unit count** —
5+ units → Large Multi-Family, 2–4 → Small Multi-Family, 1 → Other Residential; office and
mixed-use condo complexes go to Office / Commercial Condo and Mixed Use respectively. The
standalone `Condominium` category is retired. Fully-exempt parcels (zero taxable land
**and** improvement — churches, public schools, government, parks, all-exempt complexes)
are flagged and held out of the revenue-neutral solver. The standard
`$0-improvement → Vacant Land` override is applied to taxable parcels only; fully-exempt
parcels are labeled `Exempt`.

In [ ]:
def categorize_property_type(desc: str) -> str:
    """Map King County PREUSE_DESC to a standard LVTShift property category."""
    if pd.isna(desc):
        return 'Other'
    d = str(desc).strip().lower()
    if d == '':
        return 'Other'
    if d.startswith('vacant') or 'vacant land' in d:
        return 'Vacant Land'
    if 'single family' in d:
        return 'Single Family Residential'
    if 'townhouse' in d or 'rowhouse' in d:
        return 'Townhome / Rowhouse'
    if d.startswith(('duplex', 'triplex', '4-plex', 'tri-plex')):
        return 'Small Multi-Family (2-4 units)'
    if 'condominium(residential)' in d:
        return 'Condominium'
    if 'condominium(mixed' in d or 'apartment(mixed' in d or 'mixed use' in d:
        return 'Mixed Use'
    if 'condominium(office)' in d:
        return 'Office / Commercial Condo'
    if d.startswith('apartment'):
        return 'Large Multi-Family (5+ units)'
    if any(k in d for k in ['rooming house', 'retirement', 'nursing home', 'group home',
                            'residence hall', 'dorm', 'fraternity', 'sorority', 'mobile home',
                            'houseboat', 'bed & breakfast', 'rehabilitation', 'historic prop(residence']):
        return 'Other Residential'
    if 'hotel' in d or 'motel' in d:
        return 'Hotel'
    if 'office' in d or 'medical/dental' in d or d.startswith('bank'):
        return 'Office / Commercial Condo'
    if any(k in d for k in ['retail', 'shopping ctr', 'grocery', 'conv store', 'restaurant',
                            'tavern', 'auto showroom', 'service station', 'gas station', 'car wash',
                            'mini lube', 'daycare', 'bowling', 'health club', 'movie theater',
                            'skating', 'art gallery', 'auditorium', 'golf', 'sport facility',
                            'driving range', 'vet/animal', 'mortuary', 'club']):
        return 'Retail / General Commercial'
    if any(k in d for k in ['warehouse', 'industrial', 'high tech', 'high flex', 'terminal(',
                            'air terminal', 'shell structure', 'greenhse']):
        return 'Industrial'
    if d.startswith('parking'):
        return 'Transportation - Parking'
    if any(k in d for k in ['church', 'school(', 'park, public', 'governmental', 'utility, public',
                            'utility, private', 'post office', 'hospital', 'museum', 'open space',
                            'reserve/wilderness', 'transferable dev']):
        return 'Exempt'
    if any(k in d for k in ['tideland', 'water body', 'river/creek', 'easement', 'marina']):
        return 'Other'
    return 'Other Commercial'


gdf['PROPERTY_CATEGORY'] = gdf['PREUSE_DESC'].apply(categorize_property_type)

# Residential condominiums -> multifamily categories (Greg, 2026-07-23): an aggregated
# complex is one multifamily building stack, sorted by its unit count — 5+ units is Large
# Multi-Family, 2-4 (Seattle's ubiquitous townhome-style condo plats) is Small Multi-Family,
# a 1-unit plat is Other Residential. Rows without a unit count (the handful of non-'K'
# parcels carrying a residential-condo use code) default to Large Multi-Family per spec.
# Commercial/mixed condo complexes are already routed by use code above (Condominium(Office)
# -> Office / Commercial Condo; Condominium(Mixed Use) -> Mixed Use).
_is_condo = gdf['PROPERTY_CATEGORY'] == 'Condominium'
_u = gdf['condo_units']
gdf.loc[_is_condo & ((_u >= 5) | _u.isna()), 'PROPERTY_CATEGORY'] = 'Large Multi-Family (5+ units)'
gdf.loc[_is_condo & _u.between(2, 4), 'PROPERTY_CATEGORY'] = 'Small Multi-Family (2-4 units)'
gdf.loc[_is_condo & (_u == 1), 'PROPERTY_CATEGORY'] = 'Other Residential'
assert (gdf['PROPERTY_CATEGORY'] != 'Condominium').all(), "Unrouted Condominium rows remain"

# Fully-exempt = zero taxable land AND zero taxable improvement
gdf['full_exmp'] = (gdf['taxable_total'] <= 0).astype(int)

# $0-improvement (but taxable) -> Vacant Land; then exempt label takes precedence
mask_vacant = (gdf['full_exmp'] == 0) & (gdf['taxable_improvement_value'] <= 0)
gdf.loc[mask_vacant, 'PROPERTY_CATEGORY'] = 'Vacant Land'
gdf.loc[gdf['full_exmp'] == 1, 'PROPERTY_CATEGORY'] = 'Exempt'

print(f"Fully-exempt parcels held out of solver: {gdf['full_exmp'].sum():,}")
print(gdf['PROPERTY_CATEGORY'].value_counts())

## Section 4 — Current Targeted Revenue (validation)

`current_tax` already holds the targeted-levy slice. Because rates come from the official
2025 *Codes and Levies* rate book and values from the official Assessor extract, the
reconstruction *is* the King County figure by construction. We anchor plausibility against the
published 2025 county value base (\$873B; Seattle ≈ 29%).

In [ ]:
taxable_parcels = gdf[gdf['full_exmp'] == 0]
target_revenue = float(taxable_parcels['current_tax'].sum())
print(f"Targeted voted-levy revenue (Seattle, 2025): ${target_revenue:,.0f}")
print(f"  taxable parcels: {len(taxable_parcels):,}   exempt: {int(gdf['full_exmp'].sum()):,}")
print(f"  implied as share of all-levy Seattle bill: "
      f"{target_revenue / (gdf['taxable_total']*gdf['total_rate']/1000).sum()*100:.1f}%")
assert target_revenue > 0

## Section 5 — Building-Exemption Model (revenue-neutral on the targeted slice)

`model_stacking_improvement_exemption` exempts the first \$200k of each building plus 50% of the
remainder, then sets a single millage on the **remaining land + building value** so the targeted
levies raise the same total. Because the exemption is uniform across parcels, pooling the
targeted levies is equivalent to holding each levy neutral individually (they share the same
parcel base).

In [ ]:
# Exclude fully-exempt parcels from the reform AND the modeled output (held out, not added back).
n_exempt = int((gdf['full_exmp'] == 1).sum())
taxable = gdf[gdf['full_exmp'] == 0].copy()

millage_rate, new_revenue, taxable = model_stacking_improvement_exemption(
    df=taxable,
    land_value_col='taxable_land_value',
    improvement_value_col='taxable_improvement_value',
    current_revenue=taxable['current_tax'].sum(),
    improvement_exemption_percentage=IMPROVEMENT_EXEMPTION_PCT,   # 50% of remainder
    building_abatement_floor=BUILDING_ABATEMENT_FLOOR,           # first $200k exempt
)
gdf = taxable   # modeled parcels only — fully-exempt are excluded from export + charts

# Single uniform millage on remaining (post-exemption) value -> land and building alike
land_millage = improvement_millage = float(millage_rate)
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"\nNew uniform millage on remaining value: {millage_rate:.5f} /$1000")
print(f"  (first ${BUILDING_ABATEMENT_FLOOR:,.0f} of each building exempt + "
      f"{IMPROVEMENT_EXEMPTION_PCT*100:.0f}% of the remainder)")
print(f"vs. combined in-scope rate today: {IN_SCOPE_RATE:.5f} /$1000 on full land+building")
print(f"Revenue neutrality: ${new_revenue:,.0f} vs target ${target_revenue:,.0f} "
      f"({(new_revenue/target_revenue-1)*100:+.4f}%)")

category_summary = calculate_category_tax_summary(
    df=gdf, category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax', new_tax_col='new_tax',
)
print_category_tax_summary(category_summary, title="Seattle — 50% + $200k Building Exemption on Voted, Seattle-Contained Levies")

## Section 6 — Exploratory Chart (optional)

In [ ]:
matplotlib.use('Agg')
fig, ax = plt.subplots(figsize=(10, 6))
summary = (gdf[gdf['full_exmp'] == 0]
           .groupby('PROPERTY_CATEGORY')['tax_change_pct'].median().sort_values())
summary.plot.barh(ax=ax, color=np.where(summary >= 0, '#c0392b', '#27ae60'))
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Seattle — Median % Change by Category\n(50% + $200k Building Exemption on Voted, Seattle-Contained Levies)')
ax.set_xlabel('Median % change in targeted-levy tax')
plt.tight_layout()
plt.savefig(DATA_DIR / 'category_preview.png', dpi=150)
plt.close()
print("saved category_preview.png")

## Section 7 — Census Join + Standard Export

In [ ]:
# Census join — load cached King County block groups (fast FTP path), else fetch.
# King County (53033) has ~1,545 block groups; the TIGERweb per-tract fetch is very slow,
# so we cache via the Census FTP shapefile + ACS API (see data/census/).
import geopandas as _gpd
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

_census_dir = DATA_DIR / 'census'
_cache_gdf = _census_dir / 'king_census_gdf.parquet'
_cache_data = _census_dir / 'king_census_data.parquet'

try:
    if _cache_gdf.exists() and _cache_data.exists():
        census_gdf = _gpd.read_parquet(_cache_gdf)
        census_data = pd.read_parquet(_cache_data)
        print(f"Loaded cached King County census: {len(census_gdf):,} block groups")
    else:
        _fips = STATE_FIPS + COUNTY_FIPS
        census_data, census_gdf = get_census_data_with_boundaries(_fips, 2022)

    gdf = match_to_census_blockgroups(gdf, census_gdf)
    if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
        gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
    if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
        gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
    print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
except Exception as e:
    print(f"Census join failed: {e}")
    for _col in ['std_geoid', 'median_income', 'minority_pct', 'black_pct']:
        gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

# Standard report — PNGs in analysis/reports/seattle/
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False)
print("Done.")

In [ ]:
# Interactive parcel map — reuse the exact standard-export frame (gdf) so the map matches the CSV.
import geopandas as _gpd
from lvt.parcel_map import save_parcel_map_export, create_parcel_map

_map_gdf = gdf   # same frame passed to save_standard_export (GeoDataFrame, geometry populated)

_map_out = save_parcel_map_export(
    gdf=_map_gdf,
    city='seattle',
    output_path='../../analysis/maps/seattle.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col='PIN',
    parcel_url_template=None,
    owner_name_col=None,
    owner_address_col='ADDR_FULL',
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)
create_parcel_map(_map_out, 'seattle', simplify_tolerance_m=6)
print('seattle parcel map done.')

## Validation Summary

| Check | Result |
|---|---|
| Reform | **Building exemption: \$200k base + 50% of remainder** (uniform millage on remaining land+building value) |
| Commercial value fix | **3,581 commercial parcels** had a \$1,000 placeholder improvement restored via cost approach (+\$5.29B taxable improvement) |
| Condominium aggregation | **46,164 unit accounts** summed onto **3,094 complex land parcels** (Section 1c): +\$9.87B taxable land, +\$22.09B taxable improvements, replacing \$5.2B of token K-record value; 395 exempt units contribute \$0; 35 fully-exempt complexes held out. Complexes routed by unit count: 1,458 → Large Multi-Family (5+), 1,558 → Small Multi-Family (2–4), 77 → Other Residential; office/mixed condo complexes to their commercial categories |
| Revenue neutrality | \$608,862,990 modeled = \$608,862,990 target (+0.0000%) |
| In-scope slice | \$608.9M @ **2.08119 /\$1000** = City lid-lifts + reg lid-lift + voted GO bond (≈\$483.5M) + Seattle Metropolitan Park District (≈\$125.4M) |
| New millage | **2.91952 /\$1000 on remaining (post-exemption) value**, vs. the 2.08119 combined in-scope rate on full value today |
| Parcel count | 188,624 parcels: 185,530 `LEVY_JURIS == SEATTLE` + 3,094 condo complex land parcels (Seattle TCAs), tax year 2025 |
| Exempt held out of solver | 7,130 fully-exempt parcels (excluded from export + charts); 181,494 modeled |
| Census coverage | **100.0%** of parcels matched to block groups |
| PNGs generated | 7 of 7 |

**Commercial building-value correction (Section 1b).** The raw King County roll records a ~\$1,000
placeholder improvement for many income-producing commercial parcels (retail, office, warehouse,
restaurant, bank, …) whose building value is carried off-parcel (leasehold / personal-property
accounts). Of the ~22,700 placeholder parcels, only the **3,581 that have a real building**
(Commercial Building file square footage) **and are taxable** are revalued; the rest are genuinely
vacant land / surface parking and keep their land-only values. Revaluation is a cost approach:
building net sqft × the median \$/sqft (≈\$146 overall) of comparable Seattle buildings with both
real sqft and a real assessed value.

**Condominium aggregation (Section 1c).** Every King County condo unit is its own Real Property
account (allocated land share + unit improvement) with no GIS polygon; the complex's `PROPTYPE 'K'`
land parcel carries only a token value (≈ one average unit) and a null `LEVY_JURIS`, so earlier runs
dropped condos almost entirely (n=44 "Condominium" rows). Unit **taxable** values are now summed
onto the complex land parcel — 46,164 units → 3,094 complexes, +\$31.95B taxable value (10.9% of the
tax base) and +\$66.5M of in-scope levy — and each complex is modeled as **one parcel**: the \$200k
building floor applies **once per complex** (the model's per-parcel building-stack convention), not
per unit. No double-counting: the roll has no Minor-0000 complex account, and unit accounts that are
their own polygons in the frame (5, all fully exempt) are excluded.

**Distribution (targeted-levy tax only).** Adding condos grows the levy target 12.3% but grows the
post-exemption base slightly less — condos are more improvement-heavy (69% building) than the
citywide base (47%), so a larger share of their value is exempted — and the floated millage ticks
up from 2.88581 to **2.91952**. Existing homeowners' cuts therefore shrink slightly while condo
complexes, as heavily-built parcels, mostly land on the winning side. Medians (taxable parcels):

- *Pay more (land-heavy):* Vacant Land **+40.3%** (genuinely all land → at the ceiling), Parking **+23.3%**, Other Commercial **+21.0%**, Retail/General Commercial **+19.2%**, Office/Commercial Condo **+13.3%**, Industrial **+9.6%**, Small Multi-Family **+2.2%**, Large Multi-Family **+2.0%**.
- *Benefit / near-flat:* Single Family Residential **−2.0%** (a modest cut), Hotel **−1.0%**, Mixed Use **−5.1%**, Townhome/Rowhouse **−21.2%**.

The increase ceiling is `2.91952 / 2.08119 − 1 = +40.3%`; only genuinely land-only parcels reach it.

**Limitations.**
1. **Commercial values are imputed, not assessor-certified** — the ~3,581 corrected parcels use a cost-approach estimate (sqft × use-median \$/sqft); far more reliable than the \$1,000 placeholder, but an estimate.
2. **Condo vintage mismatch** — unit values come from the Assessor's live Real Property Account extract (bill year 2026, i.e. 2025 assessments) while the parcel layer carries tax-year-2025 values (2024 assessments); condo values are therefore one assessment cycle newer than the rest of the base.
3. **One \$200k floor per complex** — a multi-building garden-style condo complex gets a single floor, conservative relative to a per-building application.
4. **Non-polygon accounts without a condo complex parcel** — ~1,502 Seattle-TCA accounts (≈\$0.9B taxable, 0.3% of the base) have no GIS polygon and no K-parcel major (air rights, ground leases, extract gaps) and remain out of the model, as before.
5. **Senior/disabled exemptions** — reflected in taxable values where King County applies them; no per-parcel senior flag exists in the extract.
6. **Parcel base vs. certified base** — modeled slice \$608.9M vs. certified real-world collection \$600.9M (**+1.3%**, well within the 5% gate; formerly −9.7% with condos missing). The small overage is the condo vintage (one cycle newer) net of business personal property + state-assessed utilities, which sit outside the parcel file.
7. **Scope** — Seattle School District No. 1 is *excluded* (≈ coextensive with the city but not wholly contained).
8. **Frame** — `current_tax`/`new_tax` are the *targeted voted-levy slice only*, not the full property-tax bill.